In [ ]:
# Run once for imports
# !pip install pandas numpy shap pdfplumber scikit-learn joblib google-generativeai --quiet
# !pip install --upgrade google-ai-generativelanguage google-generativeai --quiet
# !pip install --upgrade scikit-learn==1.8.0 --quiet


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


# Data Preprocessing and Helper functions

In [ ]:
import shap
import joblib
import pandas as pd
import numpy as np
features = pd.read_csv('features.csv')      # features.csv is saved by the final model ipynb
kyc_individual = pd.read_csv("kyc_individual.csv")
kyc_smallbusiness = pd.read_csv("kyc_smallbusiness.csv")

In [ ]:
# split our data into individual and business features
individual_features = features[features['customer_id'].isin(kyc_individual['customer_id'])].drop(columns=['ttl_volume_to_sales','trans_count_to_sales','avg_volume_to_sales','revenue_capacity_ratio','sales'])
business_features = features[features['customer_id'].isin(kyc_smallbusiness['customer_id'])].drop(columns=['ttl_volume_to_income','trans_count_to_income','avg_volume_to_income','outbound_to_income_ratio','income'])
individual_risk_profiles = pd.read_csv('model_output_individual_accounts.csv')
business_risk_profiles = pd.read_csv('model_output_business_accounts.csv')

In [3]:
IndividualIsolationForest = joblib.load('iso_forest_individual.joblib')
BusinessIsolationForest = joblib.load('iso_forest_business.joblib')

In [4]:
def get_isolation_forest_shap(model, input_vector, feature_names=None):
    if isinstance(input_vector, list):
        input_vector = np.array(input_vector).reshape(1, -1)
    elif isinstance(input_vector, np.ndarray) and input_vector.ndim == 1:
        input_vector = input_vector.reshape(1, -1)
    
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(input_vector)
    
    if feature_names:
        # Trim shap_values to match the number of features
        # (newer SHAP versions append extra bias columns)
        shap_values = shap_values[:, :len(feature_names)]
        return pd.DataFrame(shap_values, columns=feature_names)
    
    return shap_values

In [5]:
individual_features_list = ['customer_id', 'age', 'card_min', 'account_age', 'activity_density',
       'channel_concentration', 'max_to_avg_ratio', 'card_mean_to_max_ratio',
       'ttl_volume_to_income', 'total_trans_volume', 'churn_anomaly_count',
       'disc_tf', 'trans_count_to_income', 'emt_sum', 'income',
       'card_coef_variation', 'tf_kurt', 'card_mean', 'amount_concentration',
       'eft_range', 'emt_preference', 'wire_preference']
business_features_list = ['customer_id', 'velocity_variance_7day', 'eft_median', 'account_age',
       'emt_mean', 'activity_density', 'emt_min', 'emt_preference',
       'consistency_score', 'tf_entropy', 'disc_tf', 'tf_kurt', 'card_sum',
       'max_to_avg_ratio', 'emt_std', 'revenue_capacity_ratio',
       'total_trans_volume', 'total_outbound_amount', 'card_mean_to_max_ratio',
       'avg_trans_volume', 'eft_std', 'card_mean', 'avg_volume_to_sales',
       'eft_coef_variation']

In [6]:
def get_feature_vector(customer_id: str, is_individual: bool = True):
    if is_individual:
        df = individual_features
        features = individual_features_list[1:] # Exclude 'customer_id'
    else:
        df = business_features
        features = business_features_list[1:] # Exclude 'customer_id'
    try:
        row_values = df.loc[df['customer_id'] == customer_id, features].values[0]
        return row_values.astype(float)
    except IndexError:
        print(f"Error: Customer ID {customer_id} not found.")
        return None

In [7]:
# Example usage:
customer_id = 'SYNID0200821735'
features = get_feature_vector(customer_id=customer_id, is_individual=False)

In [8]:
list(features)

[np.float64(53.24414715719062),
 np.float64(716.22),
 np.float64(775.0),
 np.float64(0.0),
 np.float64(0.7065217391304348),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(5.458264983697203e-05),
 np.float64(6.246434044193162),
 np.float64(1083.071676910818),
 np.float64(12.50389367141693),
 np.float64(106.67),
 np.float64(155.24704679514312),
 np.float64(0.0),
 np.float64(1644550010000.0002),
 np.float64(4345789.3),
 np.float64(1644550.0100000002),
 np.float64(0.9999999906252932),
 np.float64(13330.641983648336),
 np.float64(128238.88071315092),
 np.float64(106.67),
 np.float64(13330641983.648336),
 np.float64(9.590584443728028)]

In [9]:
# Test with IndividualIsolationForest model
get_isolation_forest_shap(model=BusinessIsolationForest, input_vector=features, feature_names=business_features_list[1:]).iloc[0].to_dict()

{'velocity_variance_7day': -0.437657821893327,
 'eft_median': 0.009840953103485619,
 'account_age': 0.02729717366437308,
 'emt_mean': 0.04826705562063612,
 'activity_density': -0.09312159629564887,
 'emt_min': 0.028840193999154728,
 'emt_preference': 0.11927242258269401,
 'consistency_score': 0.05525583666746147,
 'tf_entropy': 0.007933331919778705,
 'disc_tf': -1.0129862684038142,
 'tf_kurt': -0.4790972512479946,
 'card_sum': 0.05594622653401049,
 'max_to_avg_ratio': -0.7830518360451302,
 'emt_std': 0.05604810497780234,
 'revenue_capacity_ratio': -1.137084784499093,
 'total_trans_volume': -1.133124081980285,
 'total_outbound_amount': -0.7730136539772197,
 'card_mean_to_max_ratio': -0.6697212930689238,
 'avg_trans_volume': -0.3700109668329165,
 'eft_std': -0.7879759091781239,
 'card_mean': 0.04596340963074764,
 'avg_volume_to_sales': -0.7376326116189356,
 'eft_coef_variation': -0.6623587074222905}

In [10]:
from openpyxl import load_workbook
feature_explanations = load_workbook('feature_description.xlsx')['Sheet1']

In [11]:
import pandas as pd
df_desc = pd.read_excel('feature_description.xlsx', sheet_name='Sheet1')
df_desc = df_desc.ffill()
df_desc.columns
def get_feature_explanation(feature: str):
    match = df_desc[df_desc['Feature'] == feature]
    if not match.empty:
        feature_info = {'Feature Name': feature, 'Feature Description': match.iloc[0]['Description'],
                        'Feature Rationale For AML': match.iloc[0]['AML Rationale']}
        return feature_info
    return None

# Example Usage on individual feature list
for feature in individual_features_list[1:]:
    print(get_feature_explanation(feature))
    print()

{'Feature Name': 'age', 'Feature Description': 'Physical age of the customer in years', 'Feature Rationale For AML': 'Statistically assumes that older, well-established accounts and older customers are generally less likely to be engaging in money laundering.'}

{'Feature Name': 'card_min', 'Feature Description': 'The smallest single card transaction for each account', 'Feature Rationale For AML': 'Helps form the baseline statistical profile of how an account natively uses different payment channels.'}

{'Feature Name': 'account_age', 'Feature Description': 'Days since the first recorded transaction (account age)', 'Feature Rationale For AML': 'Statistically assumes that older, well-established accounts and older customers are generally less likely to be engaging in money laundering.'}

{'Feature Name': 'activity_density', 'Feature Description': 'The percentage of days the account was actively transacting.', 'Feature Rationale For AML': 'Extreme values signal a bot or dedicated money m

In [12]:
def get_risk_score(customer_id: str, is_individual: bool = True) -> float | None:
    df = individual_risk_profiles if is_individual else business_risk_profiles
    match = df.loc[df['customer_id'] == customer_id, 'risk_score']
    if match.empty:
        print(f"Customer ID {customer_id} not found.")
        return None
    return float(match.values[0])
print(get_risk_score(customer_id=customer_id, is_individual=False))

0.670743113


# Generate Responses

In [ ]:
GEMINI_API_KEY = ""      # config your gemini api here
GEMINI_MODEL = "gemini-2.5-flash-lite"
PDF_PATHS = [
    'static/FIA Red Flag Indicators.pdf',
    'static/FINTRACT ML Indicators.pdf',
    'static/FFIEC BSA_AML Manual - MONEY LAUNDERING AND TERRORIST FINANCING _RED FLAGS_.pdf',
    'static/Feature Engineering for TransactionAnomalies.pdf'
]

MAX_CHARS = 1500

In [29]:
import re
import time
import shap
import joblib
import pdfplumber
import google.generativeai as genai

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel(GEMINI_MODEL)

print(f"Gemini model ready: {GEMINI_MODEL}")

Gemini model ready: gemini-2.5-flash-lite


In [30]:
def get_shap_dict(customer_id: str, is_individual: bool = True) -> dict | None:
    vec     = get_feature_vector(customer_id, is_individual)
    if vec is None:
        return None
    model   = IndividualIsolationForest if is_individual else BusinessIsolationForest
    f_names = (individual_features_list if is_individual else business_features_list)[1:]
    return get_isolation_forest_shap(model, vec, feature_names=f_names).iloc[0].to_dict()

In [31]:
def extract_chunks_from_pdf(pdf_path: str, chunk_size: int = 400, overlap: int = 80) -> list[dict]:
    chunks, full_text = [], ""
    source = Path(pdf_path).stem
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            full_text += " " + (page.extract_text() or "")
    full_text = re.sub(r'\s+', ' ', full_text).strip()
    sentences = re.split(r'(?<=[.!?])\s+', full_text)
    buffer, buf_len, chunk_id = [], 0, 0
    for sentence in sentences:
        buffer.append(sentence)
        buf_len += len(sentence)
        if buf_len >= chunk_size:
            text = ' '.join(buffer).strip()
            if len(text) > 50:
                chunks.append({'source': source, 'chunk_id': chunk_id, 'text': text})
            chunk_id += 1
            keep, kept_len = [], 0
            for s in reversed(buffer):
                kept_len += len(s)
                keep.insert(0, s)
                if kept_len >= overlap:
                    break
            buffer, buf_len = keep, kept_len
    if buffer:
        chunks.append({'source': source, 'chunk_id': chunk_id, 'text': ' '.join(buffer).strip()})
    return chunks


all_chunks = []
for pdf_path in PDF_PATHS:
    c = extract_chunks_from_pdf(pdf_path)
    all_chunks.extend(c)
    print(f"  {Path(pdf_path).stem[:60]}: {len(c)} chunks")

corpus_texts = [c['text'] for c in all_chunks]
print(f"\nTotal corpus: {len(corpus_texts)} chunks")

  FIA Red Flag Indicators: 43 chunks
  FINTRACT ML Indicators: 122 chunks
  FFIEC BSA_AML Manual - MONEY LAUNDERING AND TERRORIST FINANC: 74 chunks
  Feature Engineering for TransactionAnomalies: 153 chunks

Total corpus: 392 chunks


In [32]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_df=0.90, min_df=1, sublinear_tf=True)
corpus_matrix = vectorizer.fit_transform(corpus_texts)
print(f"TF-IDF index: {corpus_matrix.shape[0]} chunks × {corpus_matrix.shape[1]} terms")

TF-IDF index: 392 chunks × 14283 terms


In [33]:
def retrieve_passages(query: str, top_k: int = 4) -> list[dict]:
    scores   = cosine_similarity(vectorizer.transform([query]), corpus_matrix).flatten()
    top_idxs = scores.argsort()[::-1][:top_k]
    return [{'score': float(scores[i]), **all_chunks[i]} for i in top_idxs if scores[i] > 0]


def build_query_from_shap(shap_dict: dict, top_n: int = 10) -> str:
    sorted_feats = sorted(
        [(k, v) for k, v in shap_dict.items() if not np.isnan(v)],
        key=lambda x: abs(x[1]),
        reverse=True        
    )[:top_n]
    terms = []
    for feat, _ in sorted_feats:
        exp = get_feature_explanation(feat)
        terms.append(f"{feat} {exp['Feature Rationale For AML']}" if exp else feat)
    return ' '.join(terms)
print(build_query_from_shap(get_shap_dict(customer_id, is_individual=False), top_n=10))

revenue_capacity_ratio total_trans_volume Suspicious accounts generally exhibit either massive total volumes or unnaturally high transaction frequencies. disc_tf Flags generally chaotic, uncharacteristic money movement that lacks any natural, smooth human spending pattern. eft_std Helps form the baseline statistical profile of how an account natively uses different payment channels. max_to_avg_ratio Flags accounts with sudden, massive layering spikes that deviate violently from their normal historical transaction sizes. total_outbound_amount A ratio near 1.0 indicates symmetric "pass-through" behavior—money coming in immediately leaves the account in the exact same amounts. avg_volume_to_sales Adjusts behavioral volume metrics against financial capacity, allowing the model to distinguish between a millionaire's normal spending and a student's money mule behavior. card_mean_to_max_ratio Detects if a single, massive illicit transaction heavily skews the account's standard average behavio

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  KNOWLEDGE LIBRARY INTEGRATION  (Task 1 → Task 3 bridge)
#  Loads the feature→indicator map exported from Task 1 and
#  provides helpers to resolve SHAP features to structured
#  indicator objects (with source citations & typology links).
# ═══════════════════════════════════════════════════════════════

import json

with open('feature_indicator_map.json') as f:
    FEATURE_INDICATOR_MAP = json.load(f)

# ── Indicators registry (mirrored from Task 1 aml_dashboard.jsx → INDICATORS) ──
INDICATORS_REGISTRY = {
    "IND-01": {
        "id": "IND-01", "category": "Channel Behavior",
        "name": "Extreme single-channel concentration",
        "detection": "HHI channel_concentration ≥ 0.75",
        "threshold": "HHI ≥ 0.75 (flag); HHI ≥ 0.90 (critical)",
        "typologies": ["TYP-01", "TYP-07"],
        "sources": ["FINTRAC-ML", "FATF-WIRE", "FEENG2023"],
        "modelFeature": "channel_concentration",
    },
    "IND-02": {
        "id": "IND-02", "category": "Flow Pattern",
        "name": "Near-perfect inflow/outflow symmetry (7-day window)",
        "detection": "churn_ratio ∈ [0.90, 1.10]",
        "threshold": "churn_ratio ∈ [0.90, 1.10] with ≥3 occurrences",
        "typologies": ["TYP-01"],
        "sources": ["FINTRAC-OCG", "SCORE2021"],
        "modelFeature": "churn_symmetry_count",
    },
    "IND-03": {
        "id": "IND-03", "category": "Flow Pattern",
        "name": "Statistically anomalous churn volatility",
        "detection": "churn_z_score > 3",
        "threshold": "Z-score > 3 (≥ 2 events flags the account)",
        "typologies": ["TYP-01", "TYP-03"],
        "sources": ["SCORE2021", "UNSUP2022"],
        "modelFeature": "churn_anomaly_count",
    },
    "IND-04": {
        "id": "IND-04", "category": "Structuring",
        "name": "High-frequency low-denomination transactions",
        "detection": "total_trans_count in top 5th percentile with avg_trans_volume < $9,500",
        "threshold": "Top 5% count + avg < $9,500 CAD",
        "typologies": ["TYP-02"],
        "sources": ["FINTRAC-GUC", "FINCEN-STRUCT", "PCMLTFA"],
        "modelFeature": "total_trans_count",
    },
    "IND-05": {
        "id": "IND-05", "category": "Structuring",
        "name": "Elevated transaction velocity variance",
        "detection": "velocity_variance_7day in top 10th percentile",
        "threshold": "velocity_variance_7day > 150 (normalized)",
        "typologies": ["TYP-02", "TYP-03"],
        "sources": ["JULLUM2020", "FINTRAC-GUC"],
        "modelFeature": "velocity_variance_7day",
    },
    "IND-06": {
        "id": "IND-06", "category": "Structuring",
        "name": "Multi-channel volume diversity (smurfing signature)",
        "detection": "channel_diversity ≥ 5 AND channel_concentration < 0.25",
        "threshold": "5+ active channels with HHI < 0.25",
        "typologies": ["TYP-02"],
        "sources": ["FINTRAC-GUC", "FINCEN-STRUCT", "FEENG2023"],
        "modelFeature": "channel_diversity",
    },
    "IND-07": {
        "id": "IND-07", "category": "Time-Frequency",
        "name": "Burst energy in STFT low-frequency band",
        "detection": "tf_kurt > 6.0 AND disc_time > 8.0",
        "threshold": "tf_kurt > 6.0 (burst); > 10.0 (severe burst)",
        "typologies": ["TYP-03"],
        "sources": ["TFFEAT2023", "UNSUP2022"],
        "modelFeature": "tf_kurt",
    },
    "IND-08": {
        "id": "IND-08", "category": "Time-Frequency",
        "name": "High temporal discontinuity (day-to-day volume shock)",
        "detection": "disc_time in top 10th percentile",
        "threshold": "disc_time > 9.0 (normalized)",
        "typologies": ["TYP-03"],
        "sources": ["TFFEAT2023"],
        "modelFeature": "disc_time",
    },
    "IND-09": {
        "id": "IND-09", "category": "Time-Frequency",
        "name": "Low signal entropy (scripted / rigid behavior)",
        "detection": "tf_entropy < 0.35",
        "threshold": "tf_entropy < 0.35 (suspicious); < 0.20 (critical)",
        "typologies": ["TYP-03", "TYP-01"],
        "sources": ["TFFEAT2023", "UNSUP2022"],
        "modelFeature": "tf_entropy",
    },
    "IND-10": {
        "id": "IND-10", "category": "Flow Pattern",
        "name": "Minimal economic retention (near-zero net flow)",
        "detection": "Derived from total_outbound_amount ≈ total_trans_volume",
        "threshold": "Net retention < 3% of gross volume",
        "typologies": ["TYP-01"],
        "sources": ["FINTRAC-OCG", "FATF-OCG"],
        "modelFeature": "total_outbound_amount",
    },
    "IND-11": {
        "id": "IND-11", "category": "KYC / Income Mismatch",
        "name": "Outbound volume exceeds declared income/sales (period-adjusted)",
        "detection": "outbound_to_income_ratio > 2.0 OR revenue_capacity_ratio > 2.0",
        "threshold": "Ratio > 2× (flag); > 5× (critical SAR consideration)",
        "typologies": ["TYP-04", "TYP-05"],
        "sources": ["FINTRAC-ML", "FINTRAC-GUC", "PCMLTFA", "SCORE2021"],
        "modelFeature": "outbound_to_income_ratio",
    },
    "IND-12": {
        "id": "IND-12", "category": "KYC / Income Mismatch",
        "name": "Total transaction volume inconsistent with occupation",
        "detection": "ttl_volume_to_income > 3.0 for high-risk occupation codes",
        "threshold": "ttl_volume_to_income > 3.0 for UNEMPLOYED/STUDENT/RETIRED",
        "typologies": ["TYP-04"],
        "sources": ["FINTRAC-ML", "PCMLTFA"],
        "modelFeature": "ttl_volume_to_income",
    },
    "IND-13": {
        "id": "IND-13", "category": "KYC / Income Mismatch",
        "name": "Business sales-to-transaction ratio anomaly",
        "detection": "revenue_capacity_ratio > 3.0 for low-median-sales industries",
        "threshold": "revenue_capacity_ratio > 3.0",
        "typologies": ["TYP-05"],
        "sources": ["FINTRAC-OCG", "FATF-RBA", "SCORE2021"],
        "modelFeature": "revenue_capacity_ratio",
    },
    "IND-14": {
        "id": "IND-14", "category": "Account Behavior",
        "name": "Dormant-then-active pattern",
        "detection": "sparsity_tf > 0.70 AND activity_density in bottom quartile AND recent burst",
        "threshold": "sparsity_tf > 0.70 with final-30d activity spike",
        "typologies": ["TYP-06"],
        "sources": ["FINTRAC-ML", "FINTRAC-OCG"],
        "modelFeature": "sparsity_tf",
    },
    "IND-15": {
        "id": "IND-15", "category": "Account Behavior",
        "name": "New account with disproportionate high-risk transaction volumes",
        "detection": "account_age < 90 AND (wire_sum > $50,000 OR wu_sum > $30,000)",
        "threshold": "account_age < 90 days with high-risk channel volume",
        "typologies": ["TYP-06", "TYP-07"],
        "sources": ["FINTRAC-ML", "FINTRAC-GUC", "FATF-RBA"],
        "modelFeature": "account_age",
    },
    "IND-16": {
        "id": "IND-16", "category": "Channel Behavior",
        "name": "Disproportionate Western Union / MSB usage",
        "detection": "wu_preference > 0.40",
        "threshold": "wu_preference > 0.40 (40% of volume through WU/MSB)",
        "typologies": ["TYP-07"],
        "sources": ["FINTRAC-ML", "FINTRAC-TFSB", "FATF-WIRE"],
        "modelFeature": "wu_preference",
    },
    "IND-17": {
        "id": "IND-17", "category": "Channel Behavior",
        "name": "Disproportionate international wire transfer usage",
        "detection": "wire_preference > 0.60",
        "threshold": "wire_preference > 0.60 (60% of volume via wire)",
        "typologies": ["TYP-07"],
        "sources": ["FATF-WIRE", "FINTRAC-ML", "JULLUM2020"],
        "modelFeature": "wire_preference",
    },
}

# ── Sources registry (key regulatory sources from Task 1 SOURCES) ──
SOURCES_REGISTRY = {
    "FINTRAC-GUC":  {"org": "FINTRAC", "title": "Guideline on the Characteristics of Unusual Transactions", "year": 2021},
    "FINTRAC-ML":   {"org": "FINTRAC", "title": "Money Laundering and Terrorist Financing Indicators – Financial Entities", "year": 2023},
    "FINTRAC-OCG":  {"org": "FINTRAC", "title": "Indicators: The Financing of Domestic Organized Crime Groups", "year": 2023},
    "FINTRAC-TFSB": {"org": "FINTRAC", "title": "Terrorist Financing and Sanctions Bulletins", "year": 2023},
    "FATF-RBA":     {"org": "FATF",    "title": "Risk-Based Approach Guidance for the Banking Sector", "year": 2014},
    "FATF-OCG":     {"org": "FATF",    "title": "ML/TF Vulnerabilities of Legal Persons", "year": 2010},
    "FATF-WIRE":    {"org": "FATF",    "title": "Guidance on the Risk-Based Approach – Wire Transfers (Rec. 16)", "year": 2013},
    "FINCEN-SAR":   {"org": "FinCEN",  "title": "SAR Activity Review – Trends, Tips & Issues", "year": 2022},
    "FINCEN-STRUCT":{"org": "FinCEN",  "title": "Advisory on Structuring", "year": 2019},
    "PCMLTFA":      {"org": "Government of Canada", "title": "Proceeds of Crime (Money Laundering) and Terrorist Financing Act", "year": 2023},
    "JULLUM2020":   {"org": "Academic", "title": "Detecting Money Laundering Transactions with Machine Learning (Jullum et al.)", "year": 2020},
    "TFFEAT2023":   {"org": "Academic", "title": "A Time-Frequency Based Suspicious Activity Detection for AML", "year": 2023},
    "UNSUP2022":    {"org": "Academic", "title": "Anomaly Detection Using Unsupervised Machine Learning Algorithms", "year": 2022},
    "SCORE2021":    {"org": "Academic", "title": "Developing a Scoring Model for Managing Money Laundering Transactions Using ML", "year": 2021},
    "FEENG2023":    {"org": "Academic", "title": "Feature Engineering for Transaction Anomalies", "year": 2023},
}

# ── Typology name lookup (from Task 1 TYPOLOGIES) ──
TYPOLOGIES_REGISTRY = {
    "TYP-01": "Pass-Through / Shell Entity",
    "TYP-02": "Structuring (Smurfing)",
    "TYP-03": "Transaction Burst / Layering Event",
    "TYP-04": "Unexplained Wealth / Income Mismatch",
    "TYP-05": "Business Revenue Mismatch",
    "TYP-06": "Dormant Account Activation",
    "TYP-07": "International Wire / MSB Concentration",
}


def get_indicators_for_shap_features(top_features, top_n=8):
    """
    Map the top-N SHAP features to Knowledge Library indicator IDs.
    Returns a list of (indicator_id, indicator_obj, abs_shap) tuples,
    deduplicated and ordered by feature impact magnitude.
    """
    seen_ids = set()
    matched = []
    for feat, shap_val in top_features[:top_n]:
        for ind_id in FEATURE_INDICATOR_MAP.get(feat, []):
            if ind_id not in seen_ids and ind_id in INDICATORS_REGISTRY:
                seen_ids.add(ind_id)
                matched.append((ind_id, INDICATORS_REGISTRY[ind_id], abs(shap_val)))
    matched.sort(key=lambda x: x[2], reverse=True)
    return matched


def format_indicator_context(matched_indicators):
    """
    Format a list of (id, indicator_obj, shap_magnitude) tuples into
    structured text suitable for injection into the LLM prompt.
    Each block includes detection rule, threshold, typology names, and
    full source citations exactly as defined in Task 1.
    """
    if not matched_indicators:
        return "No direct Knowledge Library indicator matches found for the flagged features."
    blocks = []
    for ind_id, ind, _ in matched_indicators:
        source_refs = "; ".join(
            f"{SOURCES_REGISTRY[s]['org']} – {SOURCES_REGISTRY[s]['title']} ({SOURCES_REGISTRY[s]['year']})"
            if s in SOURCES_REGISTRY else s
            for s in ind["sources"]
        )
        typ_refs = ", ".join(
            f"{t} ({TYPOLOGIES_REGISTRY[t]})" if t in TYPOLOGIES_REGISTRY else t
            for t in ind["typologies"]
        )
        blocks.append(
            f"[{ind_id}] {ind['name']} — {ind['category']}\n"
            f"  Detection rule : {ind['detection']}\n"
            f"  Threshold      : {ind['threshold']}\n"
            f"  Typologies     : {typ_refs}\n"
            f"  Sources        : {source_refs}"
        )
    return "\n\n".join(blocks)


print(f"Knowledge Library loaded: {len(INDICATORS_REGISTRY)} indicators, "
      f"{len(SOURCES_REGISTRY)} sources, {len(TYPOLOGIES_REGISTRY)} typologies, "
      f"{len(FEATURE_INDICATOR_MAP)} feature mappings")


In [34]:
def build_prompt_string(customer_id: str, is_individual: bool = True) -> str:
    shap_dict = get_shap_dict(customer_id, is_individual)
    if shap_dict is None:
        return "Customer data unavailable."

    # Sort by SHAP value ascending (most anomalous / negative first)
    top_features = sorted(
        [(k, v) for k, v in shap_dict.items() if not np.isnan(v)],
        key=lambda x: x[1]
    )

    # Feature lines with SHAP values and descriptions
    feature_lines = []
    for feat, shap_val in top_features:
        exp = get_feature_explanation(feat)
        if exp:
            feature_lines.append(
                f"- {feat}: SHAP={shap_val:.3f} | {exp['Feature Description']} | AML relevance: {exp['Feature Rationale For AML']}"
            )
        else:
            feature_lines.append(f"- {feat}: SHAP={shap_val:.3f}")

    # ── Task 1 Knowledge Library: map top SHAP features → indicators ──
    triggered_indicators = get_indicators_for_shap_features(top_features, top_n=8)
    indicator_context = format_indicator_context(triggered_indicators)

    # Supplementary PDF retrieval (supporting evidence only)
    query = ' '.join(
        f"{feat} {get_feature_explanation(feat)['Feature Rationale For AML']}" if get_feature_explanation(feat) else feat
        for feat, _ in top_features
    )
    passages = retrieve_passages(query, top_k=2)

    prompt = f"""
You are an AML compliance analyst writing internal risk justifications for Scotiabank's anomaly detection model.
Each customer has a risk score from 0 to 1 (0 = no suspicion, 1 = highest suspicion).

CUSTOMER: {customer_id} ({'individual' if is_individual else 'business'} account). Risk score = {get_risk_score(customer_id, is_individual)}.

FEATURES AND SHAP ANALYSIS (sorted by anomaly contribution, most anomalous first):
{chr(10).join(feature_lines)}

KNOWLEDGE LIBRARY INDICATORS TRIGGERED (from Task 1 AML Knowledge Library — primary regulatory context):
{indicator_context}

SUPPLEMENTARY PDF PASSAGES (supporting evidence):
{chr(10).join(f"- [{p['source']}] {p['text'][:250]}" for p in passages)}

Write a justification of no more than 1500 characters. Be concise with no filler.
- Reference key features by name and explain what each SHAP value indicates about the severity of the anomaly.
- For high anomaly scores, explicitly name the triggered Knowledge Library indicator (e.g., IND-01, IND-07) and cite its source document exactly as listed above (e.g., "FINTRAC – Money Laundering and Terrorist Financing Indicators – Financial Entities (2023)").
- Link the observed behaviour to a recognised AML typology from the Knowledge Library (e.g., TYP-01 Pass-Through / Shell Entity).
- For lower anomaly scores (<0.3), a shorter explanation as to why the account was not flagged is sufficient.
Output only the justification text. Plain prose only — no bullet points, no headers. In-text document citations only.
"""
    return prompt


def generate_justification(customer_id: str, is_individual: bool = True, max_chars: int = 2000) -> str:
    prompt = build_prompt_string(customer_id=customer_id, is_individual=is_individual)
    response = gemini_model.generate_content(
        prompt,
        generation_config=genai.types.GenerationConfig(
            max_output_tokens=600,
            temperature=0.2,
        )
    )
    text = response.text.strip()
    if len(text) > MAX_CHARS:
        text = text[:MAX_CHARS - 1].rsplit(' ', 1)[0] + '.'
    return text

In [35]:
TEST_ID = 'SYNID0103491795'

# Show which Knowledge Library indicators are triggered before generating
_shap = get_shap_dict(TEST_ID, is_individual=True)
_top  = sorted([(k, v) for k, v in _shap.items() if not np.isnan(v)], key=lambda x: x[1])
_triggered = get_indicators_for_shap_features(_top, top_n=8)

print("── Triggered Knowledge Library Indicators ──")
for ind_id, ind, shap_mag in _triggered:
    print(f"  {ind_id}  {ind['name']}")
    print(f"         sources: {', '.join(ind['sources'])}")
print()

justification = generate_justification(TEST_ID, is_individual=True)

print(f"Customer      : {TEST_ID}")
print(f"Length        : {len(justification)} / {MAX_CHARS} chars")
print(f"\nJustification :\n{justification}")

# ── EXPECTED OUTPUT (illustrative — actual text generated by LLM) ──
# Customer SYNID0103491795 (risk score 0.87) triggers IND-12 (Total transaction volume
# inconsistent with occupation) via ttl_volume_to_income (SHAP=-1.34), as defined in
# FINTRAC – Money Laundering and Terrorist Financing Indicators – Financial Entities (2023)
# and the Proceeds of Crime (Money Laundering) and Terrorist Financing Act (PCMLTFA, 2023).
# This is consistent with a TYP-04 (Unexplained Wealth / Income Mismatch) typology at the
# Integration stage. IND-03 (Statistically anomalous churn volatility) is further triggered
# by churn_anomaly_count (SHAP=-0.84), linked to a TYP-03 (Transaction Burst / Layering
# Event) pattern per the scoring model literature (SCORE2021; UNSUP2022).

Customer      : SYNID0103491795
Length        : 1498 / 1500 chars

Justification :
Customer SYNID0103491795 has a high risk score of 0.87, indicating significant suspicious activity. The most impactful features contributing to this score are `ttl_volume_to_income` (SHAP=-1.34) and `trans_count_to_income` (SHAP=-1.30). These negative SHAP values, despite their negative sign, indicate a strong deviation from expected norms relative to income. This suggests a disproportionately high transaction volume and frequency compared to the customer's stated income, a key indicator for potential money mule activity or structuring, aligning with general AML principles of adjusting behavioral metrics against financial capacity. `total_trans_volume` (SHAP=-1.22) further reinforces this, showing a substantial overall transaction volume. `emt_sum` (SHAP=-1.09) and `emt_preference` (SHAP=-0.59) highlight a significant volume and preference for EMT transactions, which can be a higher-risk channel. `disc_t

# Batch loading: Generate all explanations.
* Goal: Load a csv file for all the individual customers and business customers explanations of shap results
* Deliverables: individuals_explanations.csv and business_explanations (columns: customer_id, explanation)

## Businesses Customers

In [36]:
business_customer_ids = set(business_risk_profiles['customer_id'])
len(business_risk_profiles)

7641

In [ ]:
# runs 35 mins

# --- PROMPT STRING AND BATCH FILE ---
# build the prompt string and prepare a batch file for business customers

import json

def build_prompt_string(customer_id: str, is_individual: bool = True) -> str:
    shap_dict = get_shap_dict(customer_id, is_individual)
    if shap_dict is None:
        return "Customer data unavailable."

    top_features = sorted(
        [(k, v) for k, v in shap_dict.items() if not np.isnan(v)],
        key=lambda x: x[1]
    )

    feature_lines = []
    for feat, shap_val in top_features:
        exp = get_feature_explanation(feat)
        if exp:
            feature_lines.append(
                f"- {feat}: SHAP={shap_val:.3f} | {exp['Feature Description']} | AML relevance: {exp['Feature Rationale For AML']}"
            )
        else:
            feature_lines.append(f"- {feat}: SHAP={shap_val:.3f}")

    # ── Task 1 Knowledge Library: map top SHAP features → indicators ──
    triggered_indicators = get_indicators_for_shap_features(top_features, top_n=8)
    indicator_context = format_indicator_context(triggered_indicators)

    query = ' '.join(
        f"{feat} {get_feature_explanation(feat)['Feature Rationale For AML']}" if get_feature_explanation(feat) else feat
        for feat, _ in top_features
    )
    passages = retrieve_passages(query, top_k=2)

    prompt = f"""
You are an AML compliance analyst writing internal risk justifications for Scotiabank's anomaly detection model.
Each customer has a risk score from 0 to 1 (0 = no suspicion, 1 = highest suspicion).

CUSTOMER: {customer_id} ({'individual' if is_individual else 'business'} account). Risk score = {get_risk_score(customer_id, is_individual)}.

FEATURES AND SHAP ANALYSIS (sorted by anomaly contribution, most anomalous first):
{chr(10).join(feature_lines)}

KNOWLEDGE LIBRARY INDICATORS TRIGGERED (from Task 1 AML Knowledge Library — primary regulatory context):
{indicator_context}

SUPPLEMENTARY PDF PASSAGES (supporting evidence):
{chr(10).join(f"- [{p['source']}] {p['text'][:250]}" for p in passages)}

Write a justification of no more than 1500 characters. Be concise with no filler.
- Reference key features by name and explain what each SHAP value indicates about the severity of the anomaly.
- For high anomaly scores, explicitly name the triggered Knowledge Library indicator (e.g., IND-01, IND-07) and cite its source document exactly as listed above (e.g., "FINTRAC – Money Laundering and Terrorist Financing Indicators – Financial Entities (2023)").
- Link the observed behaviour to a recognised AML typology from the Knowledge Library (e.g., TYP-01 Pass-Through / Shell Entity).
- For lower anomaly scores (<0.3), a shorter explanation as to why the account was not flagged is sufficient.
Output only the justification text. Plain prose only — no bullet points, no headers. In-text document citations only.
"""
    return prompt


from tqdm import tqdm

def prepare_batch_file_optimized(customer_ids, filename="", is_individual=False):
    """
    Generates a JSONL file for the Batch API. 
    Uses a single loop to stream data to disk to save RAM.
    """
    print(f"Generating request file for {len(customer_ids)} customers...")
    
    with open(filename, "w", encoding="utf-8") as f:
        for cid in tqdm(customer_ids):
            # FIXED: Aligned this line to 12 spaces (3 levels of 4)
            prompt_text = build_prompt_string(cid, is_individual=is_individual)
            
            # FIXED: Aligned this line to match the one above
            if not prompt_text or "unavailable" in prompt_text:
                continue
                
            # The structure required by the Batch API
            request_obj = {
                "custom_id": str(cid),
                "request": {
                    "contents": [{"role": "user", "parts": [{"text": prompt_text}]}],
                    "generationConfig": {
                        "maxOutputTokens": 600,
                        "temperature": 0.2
                    }
                }
            }
            f.write(json.dumps(request_obj) + "\n")
            
    print(f"Successfully created {filename}")

# Run locally
prepare_batch_file_optimized(business_customer_ids, filename="business_60k.jsonl")

Generating request file for 7641 customers...


  0%|          | 0/7641 [00:00<?, ?it/s]

100%|██████████| 7641/7641 [34:36<00:00,  3.68it/s]  

Successfully created business_60k.jsonl


In [76]:
import os
import csv
import asyncio
from google import genai
from google.genai import types

# --- API ---
#GEMINI_API_KEY = ''     # Input GEMINI API here
client = genai.Client(api_key=GEMINI_API_KEY)

# --- CONFIGURATION ---
INPUT_FILE = r"business_60k.jsonl"               # The master input file, which is the filename
OUTPUT_CSV = "model_output_explanations_bus.csv" # The final CSV output
CONCURRENCY_LIMIT = 50                           # Concurrent requests
MODEL_NAME = "gemini-2.5-flash-lite"             # The model you are using
START_LINE = 0                               


async def fetch_explanation(sem, custom_id, prompt_text, writer):
    """Hits the live Gemini API and writes to CSV instantly."""
    async with sem:
        for attempt in range(5): # Retry loop for temporary rate limits
            try:
                # Direct, instant API call
                response = await client.aio.models.generate_content(
                    model=MODEL_NAME,
                    contents=prompt_text,
                    config=types.GenerateContentConfig(
                        max_output_tokens=600,
                        temperature=0.2
                    )
                )
                
                explanation = response.text.strip()
                writer.writerow([custom_id, explanation])
                print(f"✅ Success: {custom_id}")
                return 
                
            except Exception as e:
                error_msg = str(e)
                if "429" in error_msg or "RESOURCE_EXHAUSTED" in error_msg:
                    sleep_time = 2 ** attempt  # 1s, 2s, 4s, 8s...
                    await asyncio.sleep(sleep_time)
                else:
                    writer.writerow([custom_id, f"API Error: {error_msg}"])
                    print(f"❌ Error for {custom_id}: {error_msg}")
                    return
        
        writer.writerow([custom_id, "Failed after 5 retries due to rate limits."])


# --- PIPELINE RUNNING ---
async def run_fast_pipeline(start_line):
    print(f"Starting High-Speed API Pipeline at line {start_line}...")
    
    sem = asyncio.Semaphore(CONCURRENCY_LIMIT)
    
    # 'a' (append) mode ensures we don't overwrite if you have to stop and restart
    mode = 'w' if start_line == 1 else 'a'
    
    with open(OUTPUT_CSV, mode, encoding='utf-8', newline='') as outfile:
        writer = csv.writer(outfile)
        if start_line == 1:
            writer.writerow(['customer_id', 'explanation']) 
            
        with open(INPUT_FILE, 'r', encoding='utf-8') as infile:
            tasks = []
            
            for line_num, line in enumerate(infile, 1):
                if line_num < start_line:
                    continue 
                    
                if not line.strip():
                    continue
                    
                data = json.loads(line)
                custom_id = data.get("custom_id")
                
                # Extract the prompt from your Batch-formatted JSONL
                try:
                    prompt_text = data["request"]["contents"][0]["parts"][0]["text"]
                except (KeyError, IndexError):
                    print(f"Skipping line {line_num}: Could not parse prompt text.")
                    continue
                
                task = asyncio.create_task(fetch_explanation(sem, custom_id, prompt_text, writer))
                tasks.append(task)
            
            print(f"Loaded {len(tasks)} requests. Blasting them to the live API now...")
            await asyncio.gather(*tasks)

    print(f"\n Finished! Instant results saved to {OUTPUT_CSV}")


await run_fast_pipeline(start_line=START_LINE)

Starting High-Speed API Pipeline at line 0...
Loaded 7641 requests. Blasting them to the live API now...
✅ Success: SYNID0200804142
✅ Success: SYNID0200713821
✅ Success: SYNID0200704471
✅ Success: SYNID0200296876
✅ Success: SYNID0200161792
✅ Success: SYNID0200351449
✅ Success: SYNID0200339429
✅ Success: SYNID0200157810
✅ Success: SYNID0200905899
✅ Success: SYNID0200714380
✅ Success: SYNID0200327400
✅ Success: SYNID0200937322
✅ Success: SYNID0200096821
✅ Success: SYNID0200040291
✅ Success: SYNID0200072882
✅ Success: SYNID0200581279
✅ Success: SYNID0200528299
✅ Success: SYNID0200356062
✅ Success: SYNID0200826459
✅ Success: SYNID0200568454
✅ Success: SYNID0200427693
✅ Success: SYNID0200464860
✅ Success: SYNID0200559935
✅ Success: SYNID0200560342
✅ Success: SYNID0200488915
✅ Success: SYNID0200051580
✅ Success: SYNID0200619829
✅ Success: SYNID0200540168
✅ Success: SYNID0200772910
✅ Success: SYNID0200487593
✅ Success: SYNID0200067140
✅ Success: SYNID0200619599
✅ Success: SYNID0200307122
✅ Su

In [ ]:
# #If some entries experienced 503 Error, run this code


# import random

# async def repair_explanations():
#     # 1. Load with consistent encoding
#     df_bus = pd.read_csv('model_output_explanations_bus.csv', encoding='latin-1')
#     df_bus.columns = ['customer_id', 'explanation']

#     # 2. Identify failures more broadly
#     # This catches 'Error:', NaNs, or empty strings
#     fail_mask = df_bus['explanation'].str.contains('Error:', na=True) | df_bus['explanation'].isna()
#     failed_lookup = {str(row['customer_id']): idx for idx, row in df_bus[fail_mask].iterrows()}

#     total_to_fix = len(failed_lookup)
#     print(f"Found {total_to_fix} rows to fix.")
#     if not total_to_fix:
#         return

#     # 3. Process source file
#     with open('business_60k.jsonl', 'r', encoding='utf-8') as f:
        
#         # Initialize the progress bar tracking the exact number of entries we need to fix
#         with tqdm(total=total_to_fix, desc="Repairing Entries", unit="entry") as pbar:
            
#             for line in f:
#                 data = json.loads(line)
#                 cid = str(data.get("custom_id")) # Ensure type match (string vs int)
                
#                 if cid in failed_lookup:
#                     idx = failed_lookup[cid]
#                     prompt_text = data["request"]["contents"][0]["parts"][0]["text"]
                    
#                     success = False
#                     for attempt in range(5):
#                         try:
#                             # Call the model
#                             response = await client.aio.models.generate_content(
#                                 model=MODEL_NAME,
#                                 contents=prompt_text,
#                                 config={"max_output_tokens": 600, "temperature": 0.2}
#                             )
                            
#                             # Update DataFrame
#                             df_bus.at[idx, 'explanation'] = response.text.strip()
                            
#                             # Use tqdm.write instead of print so it doesn't break the progress bar visual
#                             tqdm.write(f"✅ Fixed: {cid}")
#                             success = True
                            
#                             # Small delay to prevent immediate 503 on the next call
#                             await asyncio.sleep(0.5) 
#                             break 
                            
#                         except Exception as e:
#                             wait = (2 ** attempt) + random.random()
#                             tqdm.write(f"⚠️ Attempt {attempt+1} failed for {cid}: {e}. Retrying in {wait:.2f}s...")
#                             await asyncio.sleep(wait)

#                     if not success:
#                         tqdm.write(f"❌ Final Failure for: {cid}")
                        
#                     # Tick the progress bar forward by 1
#                     pbar.update(1)

#     # 4. Save results
#     df_bus.to_csv('model_output_explanations_bus.csv', index=False, encoding='utf-8')
#     print("Repair complete and file saved.")

# # To run it:
# await repair_explanations()

Found 1 rows to fix.


Repairing Entries:   0%|          | 0/1 [00:02<?, ?entry/s]

✅ Fixed: SYNID0200722284


Repairing Entries: 100%|██████████| 1/1 [00:03<00:00,  3.16s/entry]


Repair complete and file saved.


In [79]:
# Clean the output file

import pandas as pd
model_output_explanations_bus = pd.read_csv('model_output_explanations_bus.csv',encoding='latin-1')

# Drop the duplicated customers
model_output_explanations_bus.drop_duplicates(subset=['customer_id'], keep='last', inplace=True)
final_count_bus = len(model_output_explanations_bus)

print('Cleaned explanations for',final_count_bus,'business customers')
model_output_explanations_bus.to_csv('model_output_explanations_bus.csv', index=False, encoding='utf-8')

Cleaned explanations for 7641 business customers


## Individual Customers

In [ ]:
individual_customer_ids = set(individual_risk_profiles['customer_id'])
len(individual_risk_profiles)

In [ ]:
# -- the chunk runs about 4-5 hrs


# --- PROMPT STRING AND BATCH FILE ---
def build_prompt_string(customer_id: str, is_individual: bool = True) -> str:
    shap_dict = get_shap_dict(customer_id, is_individual)
    if shap_dict is None:
        return "Customer data unavailable."

    top_features = sorted(
        [(k, v) for k, v in shap_dict.items() if not np.isnan(v)],
        key=lambda x: x[1]
    )

    feature_lines = []
    for feat, shap_val in top_features:
        exp = get_feature_explanation(feat)
        if exp:
            feature_lines.append(
                f"- {feat}: SHAP={shap_val:.3f} | {exp['Feature Description']} | AML relevance: {exp['Feature Rationale For AML']}"
            )
        else:
            feature_lines.append(f"- {feat}: SHAP={shap_val:.3f}")

    # ── Task 1 Knowledge Library: map top SHAP features → indicators ──
    triggered_indicators = get_indicators_for_shap_features(top_features, top_n=8)
    indicator_context = format_indicator_context(triggered_indicators)

    query = ' '.join(
        f"{feat} {get_feature_explanation(feat)['Feature Rationale For AML']}" if get_feature_explanation(feat) else feat
        for feat, _ in top_features
    )
    passages = retrieve_passages(query, top_k=2)

    prompt = f"""
You are an AML compliance analyst writing internal risk justifications for Scotiabank's anomaly detection model.
Each customer has a risk score from 0 to 1 (0 = no suspicion, 1 = highest suspicion).

CUSTOMER: {customer_id} ({'individual' if is_individual else 'business'} account). Risk score = {get_risk_score(customer_id, is_individual)}.

FEATURES AND SHAP ANALYSIS (sorted by anomaly contribution, most anomalous first):
{chr(10).join(feature_lines)}

KNOWLEDGE LIBRARY INDICATORS TRIGGERED (from Task 1 AML Knowledge Library — primary regulatory context):
{indicator_context}

SUPPLEMENTARY PDF PASSAGES (supporting evidence):
{chr(10).join(f"- [{p['source']}] {p['text'][:250]}" for p in passages)}

Write a justification of no more than 1500 characters. Be concise with no filler.
- Reference key features by name and explain what each SHAP value indicates about the severity of the anomaly.
- For high anomaly scores, explicitly name the triggered Knowledge Library indicator (e.g., IND-01, IND-07) and cite its source document exactly as listed above (e.g., "FINTRAC – Money Laundering and Terrorist Financing Indicators – Financial Entities (2023)").
- Link the observed behaviour to a recognised AML typology from the Knowledge Library (e.g., TYP-01 Pass-Through / Shell Entity).
- For lower anomaly scores (<0.3), a shorter explanation as to why the account was not flagged is sufficient.
Output only the justification text. Plain prose only — no bullet points, no headers. In-text document citations only.
"""
    return prompt



from tqdm import tqdm

def prepare_batch_file_optimized(customer_ids, filename="", is_individual=True):
    """
    Generates a JSONL file for the Batch API. 
    Uses a single loop to stream data to disk to save RAM.
    """
    print(f"Generating request file for {len(customer_ids)} customers...")
    
    with open(filename, "w", encoding="utf-8") as f:
        for cid in tqdm(customer_ids):
            # FIXED: Aligned this line to 12 spaces (3 levels of 4)
            prompt_text = build_prompt_string(cid, is_individual=is_individual)
            
            # FIXED: Aligned this line to match the one above
            if not prompt_text or "unavailable" in prompt_text:
                continue
                
            # The structure required by the Batch API
            request_obj = {
                "custom_id": str(cid),
                "request": {
                    "contents": [{"role": "user", "parts": [{"text": prompt_text}]}],
                    "generationConfig": {
                        "maxOutputTokens": 600,
                        "temperature": 0.2
                    }
                }
            }
            f.write(json.dumps(request_obj) + "\n")
            
    print(f"Successfully created {filename}")

# Run locally
prepare_batch_file_optimized(individual_customer_ids, filename="individual.jsonl")

In [ ]:
#-- the chunk runs about 40 mins


# generate explanation for every individual account 


client = genai.Client(api_key=GEMINI_API_KEY)

# --- CONFIGURATION ---
INPUT_FILE = r"individual.jsonl"                 # The master input file
OUTPUT_CSV = "model_output_explanations_ind.csv" # The final CSV output
CONCURRENCY_LIMIT = 50                           # Concurrent requests
MODEL_NAME = "gemini-2.5-flash-lite"             # The model you are using
START_LINE = 0                               


async def fetch_explanation(sem, custom_id, prompt_text, writer):
    async with sem:
        for attempt in range(5): # Retry loop for temporary rate limits
            try:
                # Direct, instant API call
                response = await client.aio.models.generate_content(
                    model=MODEL_NAME,
                    contents=prompt_text,
                    config=types.GenerateContentConfig(
                        max_output_tokens=600,
                        temperature=0.2
                    )
                )
                
                explanation = response.text.strip()
                writer.writerow([custom_id, explanation])
                print(f"✅ Success: {custom_id}")
                return 
                
            except Exception as e:
                error_msg = str(e)
                if "429" in error_msg or "RESOURCE_EXHAUSTED" in error_msg:
                    sleep_time = 2 ** attempt  # 1s, 2s, 4s, 8s...
                    await asyncio.sleep(sleep_time)
                else:
                    writer.writerow([custom_id, f"API Error: {error_msg}"])
                    print(f"❌ Error for {custom_id}: {error_msg}")
                    return
        
        writer.writerow([custom_id, "Failed after 5 retries due to rate limits."])

async def run_fast_pipeline(start_line):
    print(f"Starting High-Speed API Pipeline at line {start_line}...")
    
    sem = asyncio.Semaphore(CONCURRENCY_LIMIT)
    
    # 'a' (append) mode ensures we don't overwrite if you have to stop and restart
    mode = 'w' if start_line == 1 else 'a'
    
    with open(OUTPUT_CSV, mode, encoding='utf-8', newline='') as outfile:
        writer = csv.writer(outfile)
        if start_line == 1:
            writer.writerow(['customer_id', 'explanation']) 
            
        with open(INPUT_FILE, 'r', encoding='utf-8') as infile:
            tasks = []
            
            for line_num, line in enumerate(infile, 1):
                if line_num < start_line:
                    continue 
                    
                if not line.strip():
                    continue
                    
                data = json.loads(line)
                custom_id = data.get("custom_id")
                
                # Extract the prompt from your Batch-formatted JSONL
                try:
                    prompt_text = data["request"]["contents"][0]["parts"][0]["text"]
                except (KeyError, IndexError):
                    print(f"Skipping line {line_num}: Could not parse prompt text.")
                    continue
                
                task = asyncio.create_task(fetch_explanation(sem, custom_id, prompt_text, writer))
                tasks.append(task)
            
            print(f"Loaded {len(tasks)} requests. Blasting them to the live API now...")
            await asyncio.gather(*tasks)

    print(f"\n Finished! Instant results saved to {OUTPUT_CSV}")


await run_fast_pipeline(start_line=START_LINE)

Starting High-Speed API Pipeline at line 0...
Loaded 52677 requests. Blasting them to the live API now...
❌ Error for SYNID0101447712: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
✅ Success: SYNID0107919867
✅ Success: SYNID0102915955
✅ Success: SYNID0100272634
✅ Success: SYNID0107178760
✅ Success: SYNID0103649424
✅ Success: SYNID0103315909
✅ Success: SYNID0103964915
✅ Success: SYNID0104426983
✅ Success: SYNID0101949609
✅ Success: SYNID0104165958
✅ Success: SYNID0107620175
✅ Success: SYNID0106857392
✅ Success: SYNID0107649810
✅ Success: SYNID0102044896
✅ Success: SYNID0102243008
✅ Success: SYNID0108905077
✅ Success: SYNID0106936954
✅ Success: SYNID0109564097
✅ Success: SYNID0102660791
✅ Success: SYNID0100196866
✅ Success: SYNID0107203371
✅ Success: SYNID0100759282
✅ Success: SYNID0104258407
✅ Success: SYNID0103751586
✅ Success: SYNID010050

In [ ]:
# if some aren't generated because of API ERROR (503), run this to fix


# import pandas as pd
# import json
# import asyncio
# import random
# from tqdm import tqdm

# async def repair_explanations():
#     # 1. Load with consistent encoding
#     df_ind = pd.read_csv('model_output_explanations_ind.csv', encoding='latin-1')
#     df_ind.columns = ['customer_id', 'explanation']

#     # 2. Identify failures more broadly
#     # This catches 'Error:', NaNs, or empty strings
#     fail_mask = df_ind['explanation'].str.contains('Error:', na=True) | df_ind['explanation'].isna()
#     failed_lookup = {str(row['customer_id']): idx for idx, row in df_ind[fail_mask].iterrows()}

#     total_to_fix = len(failed_lookup)
#     print(f"Found {total_to_fix} rows to fix.")
#     if not total_to_fix:
#         return

#     # 3. Process source file
#     with open('individual.jsonl', 'r', encoding='utf-8') as f:
        
#         # Initialize the progress bar tracking the exact number of entries we need to fix
#         with tqdm(total=total_to_fix, desc="Repairing Entries", unit="entry") as pbar:
            
#             for line in f:
#                 data = json.loads(line)
#                 cid = str(data.get("custom_id")) # Ensure type match (string vs int)
                
#                 if cid in failed_lookup:
#                     idx = failed_lookup[cid]
#                     prompt_text = data["request"]["contents"][0]["parts"][0]["text"]
                    
#                     success = False
#                     for attempt in range(5):
#                         try:
#                             # Call the model
#                             response = await client.aio.models.generate_content(
#                                 model=MODEL_NAME,
#                                 contents=prompt_text,
#                                 config={"max_output_tokens": 600, "temperature": 0.2}
#                             )
                            
#                             # Update DataFrame
#                             df_ind.at[idx, 'explanation'] = response.text.strip()
                            
#                             # Use tqdm.write instead of print so it doesn't break the progress bar visual
#                             tqdm.write(f"✅ Fixed: {cid}")
#                             success = True
                            
#                             # Small delay to prevent immediate 503 on the next call
#                             await asyncio.sleep(0.5) 
#                             break 
                            
#                         except Exception as e:
#                             wait = (2 ** attempt) + random.random()
#                             tqdm.write(f"⚠️ Attempt {attempt+1} failed for {cid}: {e}. Retrying in {wait:.2f}s...")
#                             await asyncio.sleep(wait)

#                     if not success:
#                         tqdm.write(f"❌ Final Failure for: {cid}")
                        
#                     # Tick the progress bar forward by 1
#                     pbar.update(1)

#     # 4. Save results
#     df_ind.to_csv('model_output_explanations_ind.csv', index=False, encoding='utf-8')
#     print("Repair complete and file saved.")

# # To run it:
# await repair_explanations()

Found 0 rows to fix.


In [ ]:
# Clean the output file

model_output_explanations_ind = pd.read_csv('model_output_explanations_ind.csv',encoding='latin-1')

# Drop the duplicated customers
model_output_explanations_ind.drop_duplicates(subset=['customer_id'], keep='last', inplace=True)
final_count = len(model_output_explanations_ind)

print('Cleaned explanations for ',final_count,'individual customers')
model_output_explanations_ind.to_csv('model_output_explanations_ind.csv', index=False, encoding='utf-8')

Cleaned explanations for  52677 individual customers
